# 03 — La diffusion de l'innovation : courbes logistiques et prévision

**Objectif** : modéliser comment un nouveau médicament conquiert son marché, et
tester si ce modèle **prédit correctement** les années qu'on ne lui montre pas —
pas seulement s'il s'ajuste bien aux données déjà observées.

**Produits étudiés** : Trulicity, Verzenio, Jardiance — les trois produits en
phase de croissance du portefeuille (notebook 01). Cymbalta est exclu : c'est un
produit en déclin, pas en diffusion.

**Méthode** : on modélise le **niveau annuel** de remboursement (proxy du nombre
de patients traités chaque année) par une courbe logistique de saturation —
pas les ventes cumulées, qui ne feraient qu'augmenter indéfiniment même après
la stabilisation du marché.


## 0. Imports et chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)  # reproductibilité du bootstrap

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

COULEURS = {'Trulicity': '#0072B2', 'Verzenio': '#D55E00', 'Jardiance': '#009E73'}
PRODUITS = ['Trulicity', 'Verzenio', 'Jardiance']

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_FILE   = PROJECT_DIR / 'data' / 'eli_lilly.csv'
OUTPUTS_DIR = PROJECT_DIR / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_FILE)

series = {}
for p in PRODUITS:
    g = df[df['nom_lilly'] == p].groupby('annee')['rem'].sum() / 1e6
    series[p] = g
    print(f'{p:10} : {g.index.min()}–{g.index.max()}  ({len(g)} points)  '
          f'{g.iloc[0]:.1f} → {g.iloc[-1]:.1f} M€')

---
## 1. Le modèle logistique

$$y(t) = \frac{L}{1 + e^{-k(t - t_0)}}$$

- **$L$** : plafond de marché (remboursement annuel à saturation, en M€)
- **$k$** : vitesse d'adoption
- **$t_0$** : point d'inflexion (année de croissance la plus rapide)

Ce modèle n'est **fiable que si les données couvrent déjà le début du plateau** —
sinon, $L$ n'est pas identifiable : on ne peut pas déduire un plafond d'une
courbe qui n'a montré que de la croissance.

In [ ]:
def logistique(t, L, k, t0):
    return L / (1 + np.exp(-k * (t - t0)))

def ajuster(annees, valeurs):
    """Ajuste la logistique ; renvoie (popt, pcov, t_relatif)."""
    t_rel = annees - annees[0]
    p0 = [valeurs.max() * 1.3, 0.5, t_rel.mean()]
    bounds = ([valeurs.max() * 0.9, 0.01, -5], [valeurs.max() * 20, 5, 20])
    popt, pcov = curve_fit(logistique, t_rel, valeurs, p0=p0, maxfev=20000, bounds=bounds)
    return popt, pcov, t_rel

resultats = {}
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

for ax, p in zip(axes, PRODUITS):
    g = series[p]
    annees, valeurs = g.index.values.astype(float), g.values
    popt, pcov, t_rel = ajuster(annees, valeurs)
    perr = np.sqrt(np.diag(pcov))
    resultats[p] = {'popt': popt, 'pcov': pcov, 'annees': annees, 'valeurs': valeurs}

    t_fin = np.linspace(t_rel.min(), t_rel.max(), 200)
    ax.plot(annees[0] + t_fin, logistique(t_fin, *popt), color=COULEURS[p], linewidth=2)
    ax.scatter(annees, valeurs, color=COULEURS[p], s=45, zorder=3, edgecolors='white')
    ax.axhline(popt[0], color=COULEURS[p], linestyle=':', linewidth=1, alpha=0.6)
    ax.set_title(f'{p}\nL={popt[0]:.0f}±{perr[0]:.0f} M€  k={popt[1]:.2f}', fontsize=10)
    ax.set_xlabel('Année')

axes[0].set_ylabel('Remboursements annuels (M€)')
plt.tight_layout()
plt.show()

print('Paramètres estimés (données complètes) :')
for p in PRODUITS:
    popt = resultats[p]['popt']
    perr = np.sqrt(np.diag(resultats[p]['pcov']))
    annee_pic = resultats[p]['annees'][0] + popt[2]
    n = len(resultats[p]['annees'])
    print(f'  {p:10} n={n}  L={popt[0]:6.1f}±{perr[0]:5.1f} M€  '
          f'k={popt[1]:.3f}  pic de croissance ≈ {annee_pic:.1f}')

**Lecture.** Trulicity affiche un plateau net et un $L$ bien identifié
(erreur relative faible). Verzenio est encore en pleine croissance : le plafond
estimé est plus incertain. **Jardiance n'a que 5 points de données** pour 3
paramètres — l'erreur standard affichée est optimiste (voir section 4) : le
modèle a presque assez de degrés de liberté pour interpoler parfaitement les
points, ce qui ne veut pas dire que le plafond est fiable.

---
## 2. Validation temporelle : le modèle prédit-il l'avenir, ou décrit-il juste le passé ?

On ré-ajuste la courbe en **cachant les 2 dernières années**, puis on regarde si
le modèle les aurait bien prédites. Jardiance est exclu de ce test : avec
seulement 3 points d'entraînement restants pour 3 paramètres, l'ajustement
serait dégénéré.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
n_test = 2

for ax, p in zip(axes, ['Trulicity', 'Verzenio']):
    annees, valeurs = resultats[p]['annees'], resultats[p]['valeurs']
    t_rel = annees - annees[0]
    t_train, y_train = t_rel[:-n_test], valeurs[:-n_test]
    t_test,  y_test  = t_rel[-n_test:], valeurs[-n_test:]

    p0 = [y_train.max() * 1.3, 0.5, t_train.mean()]
    bounds = ([y_train.max() * 0.9, 0.01, -5], [y_train.max() * 20, 5, 20])
    popt_train, _ = curve_fit(logistique, t_train, y_train, p0=p0, maxfev=20000, bounds=bounds)
    pred_test = logistique(t_test, *popt_train)
    mape = np.mean(np.abs((pred_test - y_test) / y_test)) * 100

    t_fin = np.linspace(t_rel.min(), t_rel.max() + 0.3, 200)
    ax.plot(annees[0] + t_fin, logistique(t_fin, *popt_train), color=COULEURS[p],
            linewidth=2, linestyle='--', label='Ajusté sur train, extrapolé')
    ax.scatter(annees[:-n_test], y_train, color=COULEURS[p], s=50, zorder=3,
               edgecolors='white', label='Entraînement')
    ax.scatter(annees[-n_test:], y_test, color='#333333', marker='D', s=60, zorder=4,
               label='Réel (masqué à l\'ajustement)')
    ax.scatter(annees[-n_test:], pred_test, color=COULEURS[p], marker='x', s=90,
               zorder=4, label='Prédit')

    ax.set_title(f'{p} — MAPE test = {mape:.0f} %')
    ax.set_xlabel('Année')
    ax.legend(fontsize=7.5, loc='upper left')

    print(f'{p}: prédit {annees[-n_test:].astype(int)} = {pred_test.round(0)}  '
          f'|  réel = {y_test.round(0)}  |  MAPE = {mape:.1f} %')

axes[0].set_ylabel('Remboursements annuels (M€)')
plt.tight_layout()
plt.show()

**Lecture — le résultat le plus intéressant du notebook.** Pour Verzenio, le
modèle entraîné sans les deux dernières années prédit correctement la suite
(erreur faible) : la croissance suit la trajectoire attendue.

Pour **Trulicity**, c'est l'inverse : le modèle, entraîné sur 2016–2023, **sur-
estime** nettement 2024 et 2025. Autrement dit, Trulicity a ralenti **plus vite**
que ne le suggérait sa propre trajectoire passée. C'est un premier indice
quantitatif — pas une preuve — que quelque chose d'externe freine sa croissance
au-delà de la simple saturation du marché du diabète. Le candidat le plus
probable, absent de nos données (voir limites) : la **cannibalisation par
Mounjaro** (tirzépatide), lancé sur la même indication et dont les
remboursements ne sont pas dans ce dataset.

---
## 3. Projection 2026–2027 avec intervalles (bootstrap sur les résidus)

On ré-échantillonne les résidus du modèle ajusté sur **toutes** les données
disponibles, on ré-ajuste à chaque tirage, et on projette 2026–2027. La
dispersion des projections donne un intervalle empirique — à interpréter avec
prudence (voir mise en garde ci-dessous).

In [ ]:
def bootstrap_projection(annees, valeurs, popt, horizon=(1, 2), B=500):
    t_rel = annees - annees[0]
    residus = valeurs - logistique(t_rel, *popt)
    bounds = ([valeurs.max() * 0.9, 0.01, -5], [valeurs.max() * 20, 5, 20])
    t_futur = np.array([t_rel[-1] + h for h in horizon])

    projections = []
    for _ in range(B):
        y_boot = logistique(t_rel, *popt) + np.random.choice(residus, size=len(residus), replace=True)
        try:
            popt_b, _ = curve_fit(logistique, t_rel, y_boot, p0=popt, maxfev=20000, bounds=bounds)
            projections.append(logistique(t_futur, *popt_b))
        except RuntimeError:
            continue
    return np.array(projections)

lignes = []
fig, ax = plt.subplots(figsize=(10, 6))
offset = {'Trulicity': -0.15, 'Verzenio': 0, 'Jardiance': 0.15}

for p in PRODUITS:
    annees, valeurs, popt = resultats[p]['annees'], resultats[p]['valeurs'], resultats[p]['popt']
    projections = bootstrap_projection(annees, valeurs, popt)
    derniere_annee = int(annees[-1])

    for i, h in enumerate([1, 2]):
        annee_cible = derniere_annee + h
        p5, p50, p95 = np.percentile(projections[:, i], [5, 50, 95])
        lignes.append({'produit': p, 'annee': annee_cible, 'mediane': p50, 'ic5': p5, 'ic95': p95})
        ax.errorbar(annee_cible + offset[p], p50, yerr=[[p50 - p5], [p95 - p50]],
                    fmt='o', color=COULEURS[p], capsize=4, markersize=7,
                    label=p if h == 1 else None)
    # Historique
    ax.plot(annees, valeurs, color=COULEURS[p], alpha=0.4, linewidth=1.5)

ax.set_xlabel('Année')
ax.set_ylabel('Remboursements (M€)')
ax.set_title('Projections 2026–2027 avec intervalle empirique (90 %, bootstrap résidus)')
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.show()

projections_df = pd.DataFrame(lignes)
projections_df['largeur_ic'] = projections_df['ic95'] - projections_df['ic5']
projections_df.round(1)

**Mise en garde méthodologique — à ne pas sauter.** L'intervalle bootstrap
mesure uniquement le bruit résiduel autour de la courbe ajustée. Pour Jardiance
(5 points, 3 paramètres), le modèle interpole quasiment les données : les
résidus sont minuscules, donc l'intervalle bootstrap paraît étroit — mais cela
**ne signifie pas** que le plafond de marché est bien identifié. C'est un
intervalle optimiste qui ignore l'incertitude structurelle (le modèle
lui-même pourrait être mal spécifié à ce stade précoce). Pour Jardiance, mieux
vaut lire ces projections comme une simple **extrapolation de tendance**, pas
comme une prévision de plafond fiable.

---
## 4. Discussion : le ralentissement de Trulicity — saturation ou concurrence ?

Deux hypothèses, non exclusives, pour expliquer le tassement 2023→2024 :

1. **Saturation réelle du marché** : la population de patients diabétiques
   éligibles au dulaglutide en France est en grande partie déjà traitée —
   la courbe logistique elle-même prédit ce plateau.
2. **Cannibalisation par Mounjaro** (tirzépatide) : lancé sur la même indication
   avec un profil d'efficacité supérieur en essais cliniques, potentiellement
   plus rentable pour Lilly par patient. Absent de nos données (non remboursé
   sur la période, cf. README), son succès expliquerait pourquoi Trulicity
   ralentit *plus vite* que son propre historique ne le suggérait
   (section 2).

Les deux hypothèses ne sont pas contradictoires : un plateau structurel du
marché du diabète **et** une redistribution vers un produit plus récent du
même laboratoire peuvent coexister. Nos données ne permettent pas de trancher
— il faudrait les remboursements Mounjaro, qu'Open Medic ne couvre pas encore
sur cette période.

---
## 5. Synthèse et export

**Ce qu'il faut retenir :**
- Le modèle logistique s'ajuste bien **a posteriori** aux trois produits, mais
  sa capacité prédictive dépend de la maturité du produit : fiable pour
  Trulicity et Verzenio, à peine indicative pour Jardiance.
- La validation temporelle révèle que **Trulicity ralentit plus vite que prévu**
  — signal cohérent avec une cannibalisation par Mounjaro, invisible dans ce
  dataset.
- Les intervalles de projection doivent être lus avec leurs limites : un
  intervalle étroit ne garantit pas une estimation fiable quand les degrés de
  liberté sont faibles (Jardiance).

In [ ]:
export_params = pd.DataFrame([
    {'produit': p, 'n_points': len(resultats[p]['annees']),
     'L_M_euros': resultats[p]['popt'][0], 'k': resultats[p]['popt'][1],
     'annee_pic_croissance': resultats[p]['annees'][0] + resultats[p]['popt'][2]}
    for p in PRODUITS
])

export_params.to_csv(OUTPUTS_DIR / 'diffusion_parametres.csv', index=False, encoding='utf-8')
projections_df.to_csv(OUTPUTS_DIR / 'diffusion_projections.csv', index=False, encoding='utf-8')
print('Exporté : diffusion_parametres.csv et diffusion_projections.csv')
export_params.round(2)